# 03. 생성 이미지 평가 하네스

목표: 선별된 예시의 인상에 의존하지 않고, 텍스트 정확성·레이아웃·사실성·세부 표현·편집 보존성을 분리해 후보를 비교합니다. 여기서는 가상의 평가값으로 계산 구조를 실습합니다.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Evaluation:
    candidate: str
    text_accuracy: float
    layout_accuracy: float
    factual_accuracy: float
    detail_quality: float
    edit_preservation: float
    latency_seconds: float
    hard_failures: tuple[str, ...] = ()

## 문자열 정확성

OCR 결과가 있다면 정답 문자열과 편집 거리를 비교할 수 있습니다. 아래 구현은 짧은 레이블을 위한 순수 Python Levenshtein distance입니다.

In [ ]:
def edit_distance(expected, observed):
    previous = list(range(len(observed) + 1))
    for row, expected_char in enumerate(expected, start=1):
        current = [row]
        for col, observed_char in enumerate(observed, start=1):
            insert = current[col - 1] + 1
            delete = previous[col] + 1
            replace = previous[col - 1] + (expected_char != observed_char)
            current.append(min(insert, delete, replace))
        previous = current
    return previous[-1]

def normalized_text_accuracy(pairs):
    scores = []
    for expected, observed in pairs:
        denominator = max(len(expected), len(observed), 1)
        scores.append(1 - edit_distance(expected, observed) / denominator)
    return sum(scores) / len(scores) if scores else 0.0

ocr_pairs = [("확산 모델 입문", "확산 모델 입문"), ("반복 복원", "반복 복원")]
print("text accuracy:", normalized_text_accuracy(ocr_pairs))

## 가중 점수와 hard gate

평균 점수가 높아도 의료 수치 오류나 인물 정체성 훼손처럼 허용할 수 없는 실패가 있으면 탈락시킵니다. 가중치는 프로젝트 목적에 맞게 사전에 고정해야 합니다.

In [ ]:
WEIGHTS = {
    "text_accuracy": 0.30,
    "layout_accuracy": 0.25,
    "factual_accuracy": 0.20,
    "detail_quality": 0.15,
    "edit_preservation": 0.10,
}

def weighted_score(result):
    return sum(getattr(result, metric) * weight for metric, weight in WEIGHTS.items())

def decision(result, minimum_score=0.82):
    if result.hard_failures:
        return "FAIL", f"hard failure: {', '.join(result.hard_failures)}"
    score = weighted_score(result)
    return ("PASS", f"score={score:.3f}") if score >= minimum_score else ("FAIL", f"score={score:.3f}")

In [ ]:
candidates = [
    Evaluation("prompt_extend_off", 0.96, 0.88, 0.95, 0.82, 0.91, 11.8),
    Evaluation("prompt_extend_on", 0.87, 0.94, 0.91, 0.90, 0.88, 13.2),
    Evaluation("dense_3x3", 0.91, 0.86, 0.72, 0.93, 0.90, 15.7, ("의학 수치 오류",)),
]

for result in candidates:
    status, reason = decision(result)
    print(f"{result.candidate:20} {status:4} {reason}; latency={result.latency_seconds:.1f}s")

## Pareto 관점

품질과 지연을 하나의 임의 점수로 합치기보다, 더 빠르면서 품질도 높은 후보가 있는지 확인할 수 있습니다.

In [ ]:
def is_dominated(candidate, all_results):
    candidate_score = weighted_score(candidate)
    for other in all_results:
        if other is candidate or other.hard_failures:
            continue
        at_least_as_good = weighted_score(other) >= candidate_score and other.latency_seconds <= candidate.latency_seconds
        strictly_better = weighted_score(other) > candidate_score or other.latency_seconds < candidate.latency_seconds
        if at_least_as_good and strictly_better:
            return True
    return False

eligible = [result for result in candidates if not result.hard_failures]
frontier = [result.candidate for result in eligible if not is_dominated(result, eligible)]
print("Pareto frontier:", frontier)

실제 실험에서는 prompt, seed, 모델 ID, 옵션, 원본·참조 이미지 hash, 생성 시각과 평가자 정보를 함께 저장하세요. 적은 수의 성공 예시보다 고정된 test set의 평균, 하위 분위수와 실패 유형 분포가 더 중요합니다.